In [ ]:
import logging
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, RandomizedSearchCV



logging.basicConfig(
    format="%(asctime)s.%(msecs)d %(levelname)s %(filename)s:%(lineno)d %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger()
logger.setLevel(logging.INFO)


PROJECT_ROOT = Path.cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

logging.info(f"Project root added to path: {PROJECT_ROOT}")

from src.core.feature_engine import SimpleEmbeddingEngine, LegalBertEmbeddingEngine
from src.core.classifier import ClauseClassifier, MODELS_TO_EXPERIMENT
from src.core.postprocessing import compute_bayesian_posterior


In [ ]:
#artifacts_exp="simple_embeddings"
artifacts_exp="legal_bert_embeddings"

zip_filepath = PROJECT_ROOT / "data" / "ToS.zip"
artifacts_dir = PROJECT_ROOT / "models" / artifacts_exp
models_output_dir = artifacts_dir / "classifiers"

ARTIFACT_EXPERIMENTS_DICT = {
    "simple_embeddings":SimpleEmbeddingEngine(),
    "legal_bert_embeddings":LegalBertEmbeddingEngine(),
}


In [ ]:
logging.info("Instantiate a fresh engine instance")
search_engine = ARTIFACT_EXPERIMENTS_DICT[artifacts_exp]

logging.info(f"Loading precomputed embeddings from {artifacts_dir}...")
search_engine.load_artifacts(artifacts_dir)


logging.info("Loading pretrained SVC_RBF_Pipeline model...")
clause_clf = ClauseClassifier()
mlp_model_dir = models_output_dir / "SVC_RBF_Pipeline"
clause_clf.load_model(mlp_model_dir)



In [ ]:
top_k = 5

X = search_engine.corpus_embeddings
x_text = search_engine.metadata_df['text'].values  # .values for safe array indexing
y = search_engine.metadata_df['is_unfair'].values

X_train, X_test, y_train, y_test, text_train, text_test = train_test_split(
    X, y, x_text, test_size=0.2, random_state=42, stratify=y
)

# Extract properly calibrated probabilities from Platt Scaling
decision_scores = clause_clf.predict_proba(X_test)
unfair_probabilities = decision_scores[:, 1]

y_pred_original = []
y_pred_bayesian = []

# Loop through test partition
for isel, clause in enumerate(text_test):
    model_prior = unfair_probabilities[isel]
    
    # FIX: Changed clause_text to clause to match the loop iterator!
    raw_search_results = search_engine.search(clause, top_k=top_k + 1)
    
    # Filter out the exact matching sentence to mitigate data leakage
    filtered_results = [
        res for res in raw_search_results 
        if res.get('text', '').strip().lower() != clause.strip().lower()
    ]
    
    # Slice down to your target top_k elements
    search_results = filtered_results[:top_k]    
    
    # Compute the Bayesian update
    updated_posterior = compute_bayesian_posterior(model_prior, search_results)
    
    # Map probabilities to hard binary verdicts (0 or 1) using the 0.5 threshold
    original_verdict = 1 if model_prior >= 0.5 else 0
    bayesian_verdict = 1 if updated_posterior >= 0.5 else 0
    
    # Append to tracking vectors
    y_pred_original.append(original_verdict)
    y_pred_bayesian.append(bayesian_verdict)

# Convert lists to clean numpy arrays matching y_test type structure
y_pred_original = np.array(y_pred_original)
y_pred_bayesian = np.array(y_pred_bayesian)

# Generate the metrics payloads as dictionaries
report_original = classification_report(y_test, y_pred_original, output_dict=True)
report_bayesian = classification_report(y_test, y_pred_bayesian, output_dict=True)

ensemble_metrics = {
    "Model": "SVC_RBF_Bayesian",
    "macro_f1": report_bayesian["macro avg"]["f1-score"],
    "macro_precision": report_bayesian["macro avg"]["precision"],
    "macro_recall": report_bayesian["macro avg"]["recall"],
    "accuracy": report_bayesian["accuracy"]
}

In [ ]:
# Quick print to check macro-averages or specific metrics (like Unfair class recall/precision)
print("Original SVC 'Unfair' F1-Score:", report_original["macro avg"]['f1-score'])
print("Bayesian Hybrid 'Unfair' F1-Score:", report_bayesian["macro avg"]['f1-score'])
